# Session 14 · Case study: tokens and semantic similarity

**Block 1 practice.** Tokenise descriptions of goods and compare token counts across languages; compute semantic similarity between descriptions of the same goods in different languages.

Needs `sentence-transformers` and `tiktoken` (see the session README). The first run downloads `intfloat/multilingual-e5-small` (about 470 MB). No API key needed.

Theory: [01-from-words-to-vectors.md](../theory/01-from-words-to-vectors.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
print(decisions.shape)

In [ ]:
import tiktoken
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
sentencepiece = model.tokenizer                     # tokeniser of the embedding model (XLM-R type)
bpe = tiktoken.get_encoding("o200k_base")           # tokeniser of GPT-4o-type models
print("input limit of the embedding model:", model.max_seq_length, "tokens")

## 1. One description, three ways of splitting it

In [ ]:
text = decisions.loc[(decisions["language"] == "de") & decisions["description"].str.len().between(80, 160),
                     "description"].iloc[3]
print(text)
print("words        ", len(text.split()), text.split()[:12])
print("SentencePiece", len(sentencepiece.tokenize(text)), sentencepiece.tokenize(text)[:14])
print("BPE          ", len(bpe.encode(text)), [bpe.decode([i]) for i in bpe.encode(text)][:14])

## 2. Token counts of 2,000 descriptions by language

In [ ]:
sample = decisions.sample(2000, random_state=0)
counts = pd.DataFrame({
    "language": sample["language"],
    "characters": sample["description"].str.len(),
    "words": sample["description"].str.split().str.len(),
    "sentencepiece": sample["description"].map(lambda t: len(sentencepiece.tokenize(t))),
    "bpe": sample["description"].map(lambda t: len(bpe.encode(t))),
})
print("tokens per word:", (counts[["sentencepiece", "bpe"]].sum() / counts["words"].sum()).round(2).to_dict())
print("share longer than the model limit:", round((counts["sentencepiece"] > model.max_seq_length - 2).mean(), 3))
by_lang = counts.groupby("language")[["characters", "sentencepiece", "bpe"]].sum()
by_lang["chars_per_bpe_token"] = by_lang["characters"] / by_lang["bpe"]
by_lang["n"] = counts["language"].value_counts()
by_lang.sort_values("n", ascending=False).head(8).round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for language in ["de", "fr", "en"]:
    part = counts[counts["language"] == language]
    ax.hist(part["sentencepiece"].clip(upper=800), bins=40, alpha=0.5, label=language)
ax.axvline(model.max_seq_length, color="black", ls="--", label="model limit (512)")
ax.set_xlabel("SentencePiece tokens per description (clipped at 800)")
ax.set_ylabel("descriptions")
ax.legend()
plt.show()

**Question.** Which language needs the most tokens per description, and why? What happens to the end of a description that is longer than 512 tokens when it is embedded?

## 3. Semantic similarity across languages

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = [
    "Spielzeugauto aus Kunststoff, mit Rädern",
    "Voiture jouet en matière plastique, avec roues",
    "Toy car of plastics with wheels",
    "Damenschuhe mit Oberteil aus Leder und Laufsohle aus Gummi",
    "Chaussures pour femmes, dessus en cuir, semelle en caoutchouc",
    "Ladies' shoes with textile uppers and rubber soles",
    "Mund-Nasen-Schutzmaske aus Vliesstoff zum Einmalgebrauch",
    "Disposable face mask made of nonwoven fabric",
]
E = model.encode(["passage: " + t for t in texts], normalize_embeddings=True)   # e5 needs a prefix
emb_sim = pd.DataFrame(E @ E.T)
tfidf_sim = pd.DataFrame(cosine_similarity(TfidfVectorizer().fit_transform(texts)))
for i, s in enumerate(texts):
    print(i, s)
emb_sim.round(2).style.background_gradient(cmap="Blues", vmin=0.7, vmax=1)

In [ ]:
tfidf_sim.round(2).style.background_gradient(cmap="Blues", vmin=0, vmax=1)

## 4. Where do embeddings and TF-IDF disagree most?

In [ ]:
pairs = []
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        pairs.append({"i": i, "j": j, "embedding": emb_sim.iloc[i, j], "tfidf": tfidf_sim.iloc[i, j]})
pairs = pd.DataFrame(pairs)
pairs["difference"] = pairs["embedding"] - pairs["tfidf"]
for _, p in pairs.sort_values("difference", ascending=False).head(3).iterrows():
    print(f"{p.embedding:.2f} vs {p.tfidf:.2f}: {texts[int(p.i)]} | {texts[int(p.j)]}")
for _, p in pairs.sort_values("difference").head(2).iterrows():
    print(f"{p.embedding:.2f} vs {p.tfidf:.2f}: {texts[int(p.i)]} | {texts[int(p.j)]}")

## Your turn

1. Sentences 3 and 5 are shoes with leather uppers, sentence 5 has textile uppers (heading 6404 instead of 6403). How similar are they according to the embeddings? What does this mean for classification with embeddings?
2. Take three real descriptions from `decisions` in languages you cannot read and one in a language you can read. Compare their similarities and read the English `keywords` to check.
3. Count the GPT-4o tokens of a 400-character Czech description and of a 400-character French one. What does this mean for API costs?

In [ ]:
# your code here